In [17]:
import json
import pandas as pd
import re
from sympy.parsing.latex import parse_latex
from sympy import Expr

In [18]:
with open("regex.txt") as file:
    commands = [line.strip() for line in file if line.strip()]

with open("forbid.txt") as file:
    forbidden = [line.strip() for line in file if line.strip()]

In [42]:
pattern_descriptors = r'\\('+ '|'.join(re.escape(cmd) for cmd in commands) + r')\{([^{}]*)\}'

NUMBER = r'(\d+\.?\d*|\.\d+)'
patterns_mult = [
    (r'\cdot', '*'),
    (r'\times', '*'),

    #multiplying with constants
    (r'(\d)([a-zA-Z])', r'\1*\2'),
    (NUMBER + r'([a-zA-Z])', r'\1*\2'),
    (NUMBER + r'\(', r'\1*('),
    (NUMBER + r'(\\[a-zA-Z])', r'\1*\2'),
    
    #implicit multiplication through parentheses
    (r'\)\(', r')*('),
    (r'\)([a-zA-Z])', r')*\1'),

    (r'(\d)(\\[a-zA-Z])', r'\1*\2')
]

def strip_descriptors(s : str) -> str:
    new = re.sub(pattern_descriptors, r'\2', s)
    if new == s:
        return s
    else:
        return strip_descriptors(new)

def strip_dollars(s: str) -> str:
    return re.sub(r'^\$\$?(.*?)\$\$?$', r'\1', s)

def strip_LR(s : str) -> str:
    s = re.sub(r'\\left\s*\.', '', s)
    s = re.sub(r'\\right\s*\.', '', s)
    s = re.sub(r'\\left\s*', '', s)
    s = re.sub(r'\\right\s*', '', s)

def handle_implicit_mult(s : str) -> str:
    for pattern, repl in patterns_mult:
        s = re.sub(pattern, repl, s)

def remove_comments(s : str) -> str:
    return re.sub(r'(?<!\\)%.*', '', s)

def remove_text(latex_str):
    result = []
    i = 0
    while i < len(latex_str):
        if latex_str[i:].startswith('\\text{'):
            i += len('\\text{')
            depth = 1
            while i < len(latex_str) and depth > 0:
                if latex_str[i] == '{':
                    depth += 1
                elif latex_str[i] == '}':
                    depth -= 1
                i += 1
        else:
            result.append(latex_str[i])
            i += 1
    return ''.join(result)

def remove_ends(s : str) -> str:
    s = re.sub(r'\\begin\{[^}]*\}', '', s)
    s = re.sub(r'\\end\{[^}]*\}', '', s)
    return s

def collapse_spaces(s : str) -> str:
    #linebreaks
    s = re.sub(r'\\\\', '', s)
    #alignment
    s = re.sub(r'&', '', s)
    #newlines
    s = re.sub(r'\n', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def remove_labels(s : str) -> str:
    s = re.sub(r'\\label\{[^}]*\}', '', s)
    return s

In [50]:
relations = {'=', '<', '>', '\\leq', '\\geq', '\\neq', '\\approx', '\\equiv', '\\sim', '\\simeq', '\\cong', '\\propto', '\\ll', '\\gg', '\\implies', '\\iff', '\\Rightarrow', '\\Leftrightarrow'}
relation_pattern = '(' + '|'.join(re.escape(r) for r in relations) + ')(?![a-zA-Z])'

def has_relation(latex_str):
    return bool(re.search(relation_pattern, latex_str))

def split_str(s : str) -> list[str]:
    parts = re.split(relation_pattern, s)
    list = [p.strip() for p in parts if p not in relations]
    return list

In [51]:
text = "\\begin{align}\nf(h_x(z')) &= E[f(z) \\mid z_S] & \\text{SHAP explanation model simplified input mapping} \\\\\n &= E_{z_{\\bar S} \\mid z_S} [f(z)] & \\text{expectation over $z_{\\bar S} \\mid z_S$} \\\\\n &\\approx E_{z_{\\bar S}} [f(z)] & \\text{assume feature independence (as in \\cite{vstrumbelj2014explaining,ribeiro2016should,shrikumar2017learning,datta2016algorithmic}) } \\label{eq:indep} \\\\\n &\\approx f([z_S,E[z_{\\bar S}]]). & \\text{assume model linearity} \\label{eq:indep_lin}\n\\end{align}"

t = remove_comments(text)
t = remove_text(t)
t = remove_ends(t)
t = collapse_spaces(t)
t = remove_labels(t)
t

"f(h_x(z')) = E[f(z) \\mid z_S] = E_{z_{\\bar S} \\mid z_S} [f(z)] \\approx E_{z_{\\bar S}} [f(z)]  \\approx f([z_S,E[z_{\\bar S}]]). "

In [52]:
split_str(t)

["f(h_x(z'))",
 'E[f(z) \\mid z_S]',
 'E_{z_{\\bar S} \\mid z_S} [f(z)]',
 'E_{z_{\\bar S}} [f(z)]',
 'f([z_S,E[z_{\\bar S}]]).']

In [ ]:
def is_invalid_expr(s : str) -> tuple[bool, str]:
    #Pattern includes anything that contains a forbidden expr
    pattern = r'\\('+ '|'.join(re.escape(term) for term in forbidden) + r')(?![a-zA-Z])'
    match = re.search(pattern, s)
    if(match):
        return True, match.group(1)
    else:
        return False, None 

In [ ]:
def parse_expression(s : str) -> Expr:
    s = strip_dollars(s)
    s = strip_descriptors(s)


In [5]:
new = strip_descriptors(s)

In [6]:
new = strip_dollars(new)

In [7]:
expr = parse_latex(new)

In [8]:
expr

y + (x + y)

In [9]:
new

'x + y + y'